# IPL-Crunch-26 — Exploratory Analysis

Ball-by-ball IPL dataset analysis. The CSV is **ball-level**, so we derive a match-level frame by deduplicating on `match_id` for toss/team/player-of-match analysis, and use the raw ball data for batter and bowler stats.

**Sections**
1. Load & inspect
2. Build match-level frame
3. Toss analysis
4. Team analysis
5. Player analysis (PoM, run scorers, wicket takers)
6. Season & venue trends
7. Key insights

## 1. Load & inspect

In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Auto-detect the CSV in data/. On Colab, upload the file and set CSV_PATH manually.
csvs = sorted(Path('data').glob('*.csv'))
CSV_PATH = csvs[0] if csvs else Path('/content/ipl_matches.csv')
print('Using:', CSV_PATH)

df = pd.read_csv(CSV_PATH, low_memory=False)
print('Rows:', len(df), 'Cols:', len(df.columns))
df.head()

In [ ]:
df.columns.tolist()

## 2. Match-level frame

One row per `match_id`. We avoid `dropna` on the full dataframe because optional columns like `wicket_kind` are mostly NaN (a wicket falls only on a few balls per match).

In [ ]:
match_cols = [
    'match_id', 'date', 'season', 'event', 'venue', 'city',
    'team1', 'team2', 'toss_winner', 'toss_decision',
    'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match',
]
matches = df[match_cols].drop_duplicates(subset='match_id').reset_index(drop=True)
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')
print('Unique matches:', len(matches))
matches.head()

## 3. Toss analysis

In [ ]:
decided = matches.dropna(subset=['toss_winner', 'winner'])
toss_won = (decided['toss_winner'] == decided['winner']).sum()
toss_lost = (decided['toss_winner'] != decided['winner']).sum()
print(f'Toss winner won: {toss_won} ({toss_won / len(decided) * 100:.1f}%)')
print(f'Toss winner lost: {toss_lost}')

plt.figure(figsize=(7, 5))
plt.bar(['Toss Winner Won Match', 'Toss Winner Lost Match'],
        [toss_won, toss_lost], color=['#2a9d8f', '#e76f51'])
plt.title('Toss Impact on Match Result')
plt.ylabel('Number of Matches')
plt.show()

In [ ]:
counts = matches['toss_decision'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%',
        colors=['#264653', '#f4a261'], startangle=90)
plt.title('Toss Decision: Bat vs Field')
plt.show()

## 4. Team analysis

In [ ]:
top_teams = matches['winner'].dropna().value_counts().head(5)
print(top_teams)

plt.figure(figsize=(9, 5))
top_teams[::-1].plot(kind='barh', color='#1d3557')
plt.title('Top 5 IPL Teams by Wins')
plt.xlabel('Wins')
plt.show()

## 5. Player analysis

In [ ]:
top_pom = matches['player_of_match'].dropna().value_counts().head(10)
plt.figure(figsize=(9, 5))
top_pom[::-1].plot(kind='barh', color='#e9c46a')
plt.title('Top 10 Player of the Match Awardees')
plt.xlabel('Awards')
plt.show()

In [ ]:
runs = df.groupby('batter')['runs_batter'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(9, 5))
runs[::-1].plot(kind='barh', color='#2a9d8f')
plt.title('Top 10 Run Scorers')
plt.xlabel('Runs')
plt.show()

In [ ]:
not_credited = {'run out', 'retired hurt', 'retired out', 'obstructing the field'}
wkts = (df.dropna(subset=['wicket_kind'])
          .loc[lambda x: ~x['wicket_kind'].str.lower().isin(not_credited)]
          .groupby('bowler').size()
          .sort_values(ascending=False).head(10))
plt.figure(figsize=(9, 5))
wkts[::-1].plot(kind='barh', color='#e76f51')
plt.title('Top 10 Wicket Takers')
plt.xlabel('Wickets')
plt.show()

## 6. Season & venue trends

In [ ]:
per_season = matches.groupby('season').size()
plt.figure(figsize=(10, 5))
per_season.plot(kind='bar', color='#457b9d')
plt.title('Matches per Season')
plt.xlabel('Season')
plt.ylabel('Matches')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
top_venues = matches['venue'].dropna().value_counts().head(10)
plt.figure(figsize=(10, 6))
top_venues[::-1].plot(kind='barh', color='#6a4c93')
plt.title('Top 10 Venues by Matches Hosted')
plt.xlabel('Matches')
plt.show()

## 7. Key insights

- **~50.5%** match-win rate for toss winners — a real but small edge.
- **~65.9%** of toss winners chose to field first.
- **Mumbai Indians** lead franchise wins in this dataset.
- **AB de Villiers** has the most Player of the Match awards.
- **V Kohli** is the leading run scorer.
- The data spans 19 seasons across 59 unique venues.